# 09d-ownership — Vì sao các lớp mới sở hữu cụm như vậy, và camera có chia cụm không

**Câu hỏi**: với nghiệm k-means của pipeline ở bước 09, lớp nào sở hữu cụm nào và vì sao (hiệu ứng đồng đều, mục tiêu SSE hay tối ưu); và trong từng lớp, cụm có bám camera hay cuộn không? (Q09.13; QD24, QD26)

**Vị trí**: phân tích (`lab/README.vi.md` §4 loại 3), dựa trên `09-cluster` (`8982942`) và cache landscape của nó; không đổi kết quả của bước 09. Đọc nhãn sau tiền đăng ký E09.07 (`1ce3233`); cell 0 dừng nếu thứ tự này sai.

**Quy tắc nhãn (QD26)**: chỉ dùng nhãn để mô tả nghiệm của pipeline và đo trần của mục tiêu k-means (nghiệm oracle). Không chấm bằng nhãn các cực tiểu khác của landscape; đó là việc của E09.07.

**Kế hoạch, chốt trước khi chạy** (2026-11-04):
1. Kiểm và đầu vào: lớp, camera, cuộn, vai trò của từng ảnh pool; `score09` trùng khoá `k5_*` của E09.05 v2.
2. Thành phần cụm: bảng cụm × lớp, sở hữu cụm, 50 ảnh cũ rơi vào đâu, ảnh bị xếp vào cụm của lớp khác có hộp lỗi của lớp chủ cụm không; hiệu ứng đồng đều (CV cỡ lớp so với CV cỡ cụm; Xiong, Wu & Chen 2009).
3. Sổ Ward: để một lớp nhỏ có cụm riêng, phải gộp hai cụm (tốn ΔJ = n_a·n_b/(n_a + n_b)·‖μ_a − μ_b‖²; Ward 1963) và tách lớp đó khỏi các cụm đang chứa nó (lợi ΔJ). Tính đúng, không chạy lại Lloyd, nên chỉ là cận.
4. Nghiệm oracle: phân hoạch theo 5 lớp mới, 50 ảnh cũ gán về tâm lớp mới gần nhất; inertia J_oracle. Chạy Lloyd từ tâm 5 lớp, được J_basin và ARI(basin, oracle) trên 698 ảnh lớp mới. So với J_best (inertia tốt nhất trong 10.000 lần khởi tạo đơn, `09-cluster` §3.1). Ba loại, chốt trước:
   - (a) J_basin < J_best: lỗi tối ưu. Có cực tiểu sâu hơn, gần lớp thật, mà k-means++ không tìm ra.
   - (b) J_basin ≥ J_best và ARI(basin, oracle) ≥ 0.8: lớp thật là một cực tiểu nhưng không sâu nhất. Mục tiêu SSE không ưa lớp thật; thêm khởi tạo không giúp.
   - (c) ARI(basin, oracle) < 0.8: lớp thật không phải điểm dừng của Lloyd. Lệch giữa mục tiêu và hình học dữ liệu.
   Kèm theo: nhóm nghiệm của basin trong landscape và khối lượng best-of-10 của nhóm đó.
5. Độ bất ổn từng ảnh d_i (best-of-10, `09-cluster` §3.3) theo lớp.
6. Q09.13 trong từng lớp: Cramér's V hiệu chỉnh chệch giữa cụm và camera, giữa cụm và cuộn; hoán vị phân tầng (camera trong cuộn, cuộn trong camera); bảng cuộn × camera; R² hiệu chỉnh của thành viên cụm, so với `05a`; lợi Ward khi chia silk_spot theo camera, theo cuộn, so với cách chia của k-means. Đối chứng âm: crease, rolled_pit, waist_folding.
7. Cụm crease: r0 s2 so với r0 s4 (QD24), so trên cùng một pool.

**Dự đoán chốt trước** [SUY LUẬN, xác suất chủ quan]:
- ở 15/15 ảnh chụp, chi phí gộp hai cụm lớn hơn lợi khi tách rolled_pit (khoảng 85%);
- với crease và inclusion, chi phí và lợi gần nhau, nên hai lớp này thay nhau giữ cụm thứ năm (khoảng 60%);
- oracle rơi vào loại (b) hoặc (c) ở ít nhất 2/3 FT seed (khoảng 65%);
- trong silk_spot, cụm bám cuộn ít nhất bằng bám camera (khoảng 55%);
- đối chứng âm không vượt mức nền (khoảng 80%).

### Nguồn gốc (provenance)

Mẫu cell 0 (`lab/README.vi.md` §11), cộng phép chặn của QD26: commit tiền đăng ký E09.07 `1ce3233` phải là tổ tiên của HEAD trước khi đọc nhãn.

In [ ]:
# cell 0: provenance (run first); QD26 guard: labels are read only after the E09.07 pre-registration
import os, subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "main.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path[:0] = [str(ROOT), str(ROOT / "lab")]
import replay; os.environ["XLA_FLAGS"] = replay._xla_flags()   # GPU deterministic mode before any jax import
git = lambda *a: subprocess.run(["git", *a], capture_output=True, text=True).stdout.strip()
print("commit", git("rev-parse", "--short", "HEAD"),
      "| code dirty", bool(git("status", "--porcelain", "--", ".", ":(exclude)runs", ":(exclude)notes",
                               ":(exclude)lab/*.ipynb", ":(exclude)lab/*.md")),
      "| XLA_FLAGS", os.environ["XLA_FLAGS"])
import sklearn
pre_ok = subprocess.run(["git", "merge-base", "--is-ancestor", "1ce3233", "HEAD"]).returncode == 0
print("scikit-learn", sklearn.__version__, "| E09.07 pre-registration 1ce3233 is an ancestor of HEAD:", pre_ok)
assert pre_ok, "QD26: E09.07 must be pre-registered (1ce3233) before this notebook reads labels"

## 1. Kiểm và đầu vào

### 1.1 Lớp, camera, cuộn và vai trò của từng ảnh pool

Gắn mỗi hàng `pool_x` của 15 ảnh chụp với ảnh của nó theo byte trong file đặc trưng train (như `05a`, `09-cluster` §4.1), rồi đọc đủ các cột của `data_io.images`: lớp, vai trò, camera, cuộn. Lớp đọc từ file phải trùng `true_pool_y` của ảnh chụp. Mỗi ảnh chụp có một bảng `POOL[run]`: lớp, camera, cuộn, vai trò, id cụm của pipeline.

Dự đoán:
- lớp trùng `true_pool_y` ở 15/15;
- pool r0 s4 gồm silk 420, inclusion 135, waist 94, crease 32, rolled 17, và 50 ảnh cũ, mỗi lớp cơ sở 10 ảnh (E09.04);
- vai trò: S1 698, reserve 50.

In [ ]:
# cell 1: inputs with labels (QD26: after the E09.07 pre-registration): every pool row of the 15 snapshots mapped to
#         its image (class, role, camera, coil) by its bytes in the train feature file; class checked against true_pool_y
import json, importlib
import numpy as np, pandas as pd
import data_io, landscape as ls
RUNS = [f"gc10_v2_ft_r{r}_s{s}" for r in range(3) for s in range(5)]
RUN, RUN_CREASE = "gc10_v2_ft_r0_s4", "gc10_v2_ft_r0_s2"            # QD24: illustration; crease mechanism
split_11 = json.load(open(ROOT / importlib.import_module("dataloaders.gc10").SPLIT))
NAMES = [c.split("_")[0] for c in split_11["classes"]]
img_11 = data_io.images("gc10")
tr_11 = img_11[img_11.part == "train"].set_index("row")
POOL, pos_11 = {}, {}
for run in RUNS:
    b = replay.snapshots(run)
    a = b["04"]["meta"]["args"]
    F = Path(a["data_dir"]) / a["pretrained_model_name"]
    if F not in pos_11:
        X = np.load(F / f"features-{F.name}.npy")
        pos_11[F] = {r.tobytes(): i for i, r in enumerate(X)}
    x = np.asarray(b["04"]["out"]["pool_x"])
    df = tr_11.loc[[pos_11[F][r.tobytes()] for r in x], ["path", "cls", "role", "cam", "coil"]].reset_index()
    df["y"] = np.asarray(b["04"]["out"]["true_pool_y"]).astype(int)
    df["cluster"] = np.asarray(b["09"]["out"]["ids"]).astype(int)
    POOL[run] = df
print("class from file == true_pool_y:", sum(bool((d.cls == d.y).all()) for d in POOL.values()), "/ 15 | rows per snapshot:",
      sorted({len(d) for d in POOL.values()}))
print("names:", NAMES)
d_11 = POOL[RUN]
print("r0 s4 pool per class:", d_11.y.map(lambda c: NAMES[c]).value_counts().to_dict())
print("roles:", d_11.role.value_counts().to_dict(), "| old images per base class:",
      d_11[d_11.y < 5].y.map(lambda c: NAMES[c]).value_counts().to_dict())

### 1.2 Bảng tầng 1 của 15 nghiệm pipeline

Tính lại `score09` của `lab/score.py` (sở hữu theo lớp đông nhất, kể cả lớp cũ; best(c); balanced accuracy ghép một–một; AMI; độ thuần) trên nghiệm pipeline của 15 ảnh chụp. So với các khoá `k5_*` mà E09.05 v2 đã ghi trong `metrics.json` của `gc10_e09_05_v2_r<r>_s<N>`; kèm `k5_inertia` so với inertia của ảnh chụp.

Dự đoán:
- mọi khoá trùng ở 15/15;
- sở hữu (silk, inclusion, rolled, crease, waist) như QD24: waist luôn 1 cụm, rolled không cụm nào, silk 2–3 cụm; cụm còn lại về crease hoặc inclusion.

In [ ]:
# cell 2: tier-1 scores of the 15 pipeline partitions (lab/score.py score09) against the k5_* keys written by E09.05 v2
import score
rows_12 = []
for run in RUNS:
    d = POOL[run]
    sc = score.score09(d.y.values, d.cluster.values, NAMES, n_old=5)
    m = json.load(open(next(Path("runs").glob(f"*__gc10_e09_05_v2_{run[11:]}")) / "metrics.json"))
    keys = [k for k in m if k.startswith("k5_") and k != "k5_inertia"]
    bad = [k for k in keys if not np.isclose(m[k], sc[k[3:]], rtol=0, atol=1e-12)]
    rows_12.append(dict(run=run[11:], keys=len(keys), mismatched=len(bad),
                        inertia_ok=bool(np.isclose(m["k5_inertia"], ls.pool(run)["inertia"], rtol=1e-6)),
                        bal_acc=round(sc["bal_acc_hungarian"], 3), owned_s_i_r_c_w=[sc[f"owned_{n}"] for n in NAMES[5:]],
                        best_c=round(sc["best_crease"], 2), best_i=round(sc["best_inclusion"], 2)))
print(pd.DataFrame(rows_12).to_string(index=False))